# Notebook 02: Compliance Pattern

The CCA exam tests whether you understand that PCI compliance and data redaction must be **enforced programmatically in code**, not just requested in a system prompt. This notebook shows both patterns on the same scenario — a customer who includes a credit card number in their message — so you can see whether PII reaches the audit log.

Pattern covered: **PostToolUse compliance callback with regex redaction** vs. prompt-only compliance instructions.

## Setup

In [ ]:
import sys
from pathlib import Path

# Add project root so helpers and customer_service are importable
sys.path.insert(0, str(Path(".").resolve()))

import anthropic
from helpers import compare_results, print_usage

from customer_service.data.customers import CUSTOMERS
from customer_service.data.scenarios import SCENARIOS
from customer_service.services.audit_log import AuditLog
from customer_service.services.container import ServiceContainer
from customer_service.services.customer_db import CustomerDatabase
from customer_service.services.escalation_queue import EscalationQueue
from customer_service.services.financial_system import FinancialSystem
from customer_service.services.policy_engine import PolicyEngine

In [ ]:
def make_services() -> ServiceContainer:
    """Create a fresh ServiceContainer with seed customer data."""
    return ServiceContainer(
        customer_db=CustomerDatabase(CUSTOMERS),
        policy_engine=PolicyEngine(),
        financial_system=FinancialSystem(),
        escalation_queue=EscalationQueue(),
        audit_log=AuditLog(),
    )


client = anthropic.Anthropic()
scenario = SCENARIOS["happy_path"]  # C001, $50 refund
# Inject a credit card number into the message for compliance testing
pii_message = scenario["message"] + " My card is 4111-1111-1111-1111."
print(f"Customer ID: {scenario['customer_id']}")
print(f"Message with PII: {pii_message}")

## Anti-Pattern: Prompt-Only Compliance

<div style="border-left: 4px solid #dc3545; padding: 12px 16px; background: #fff5f5; margin: 8px 0;">
<strong>What's wrong:</strong> The system prompt says "never log credit card numbers" and "redact before logging," but there is no programmatic enforcement. Claude sometimes follows this instruction, sometimes doesn't — compliance becomes probabilistic. A single missed redaction is a PCI violation.
</div>

In [ ]:
from customer_service.anti_patterns.prompt_compliance import (
    run_prompt_compliance_agent,
)

anti_services = make_services()
print("Running anti-pattern (prompt-only compliance)...")
anti_result = run_prompt_compliance_agent(client, anti_services, pii_message)
print(f"Stop reason: {anti_result.stop_reason}")
print(f"Tool calls: {[tc['name'] for tc in anti_result.tool_calls]}")

In [ ]:
# Check whether the raw card number reached the audit log
anti_logs = anti_services.audit_log.get_entries()
print(f"Audit log entries: {len(anti_logs)}")

anti_pii_leaked = False
for entry in anti_logs:
    details = entry.details
    if "4111" in details and "****" not in details:
        anti_pii_leaked = True
        print(f"PII LEAKED in audit log: {details[:120]}...")

if not anti_pii_leaked:
    print("Note: Prompt happened to redact this run (probabilistic — varies by run)")
    for entry in anti_logs:
        print(f"  Log entry: {entry.details[:80]}...")

In [ ]:
class _UsageWrapper:
    def __init__(self, u):
        self.usage = u


print_usage(_UsageWrapper(anti_result.usage))

## Correct Pattern: Programmatic Compliance via Callback

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Why this works:</strong> The PostToolUse compliance callback intercepts every <code>log_interaction</code> call and applies a regex to redact 16-digit card numbers before they reach the audit log. This is deterministic — the regex fires regardless of whether Claude remembered to redact, regardless of prompt wording, on every single call.
</div>

In [ ]:
from customer_service.agent import build_callbacks, get_system_prompt, run_agent_loop

correct_services = make_services()
callbacks = build_callbacks()
print("Running correct pattern (programmatic compliance callback)...")
correct_result = run_agent_loop(
    client,
    correct_services,
    pii_message,
    get_system_prompt(),
    callbacks=callbacks,
)
print(f"Stop reason: {correct_result.stop_reason}")
print(f"Tool calls: {[tc['name'] for tc in correct_result.tool_calls]}")

In [ ]:
# Verify that the full card number never reached the audit log
correct_logs = correct_services.audit_log.get_entries()
print(f"Audit log entries: {len(correct_logs)}")

correct_pii_safe = True
for entry in correct_logs:
    details = entry.details
    if "4111-1111-1111-1111" in details:
        correct_pii_safe = False
        print(f"PII LEAKED (unexpected): {details[:120]}")
    if "****" in details:
        print(f"Correctly redacted entry: {details[:100]}")

print(f"\nPII safely redacted in audit log: {correct_pii_safe}")
if correct_pii_safe:
    print("SUCCESS: Programmatic callback prevented PII from reaching audit log")

In [ ]:
print_usage(_UsageWrapper(correct_result.usage))

## Compare Results

In [ ]:
compare_results(
    {
        "pii_leaked": anti_pii_leaked,
        "audit_log_entries": len(anti_logs),
        "tool_calls": len(anti_result.tool_calls),
    },
    {
        "pii_leaked": not correct_pii_safe,
        "audit_log_entries": len(correct_logs),
        "tool_calls": len(correct_result.tool_calls),
    },
)

> **CCA Exam Tip:** If an exam answer says 'add PCI compliance rules to the system prompt,' it is WRONG. Compliance must be enforced programmatically through validation hooks and callbacks. System prompts provide context; code enforces rules. Prompt instructions are probabilistic guidance — a regex in a PostToolUse callback is a deterministic guarantee.

## Summary

- **Anti-pattern failure:** Prompt-only compliance is probabilistic — Claude may or may not redact PII before logging, depending on the run. Any single missed redaction is a real PCI violation.
- **Correct pattern guarantee:** PostToolUse compliance callback applies a regex to every `log_interaction` call before it writes to the audit log. The card number is replaced with `****-****-****-NNNN` (last 4 preserved) deterministically on every call.
- **Key principle:** Compliance enforcement belongs in code (callbacks), not in natural language (system prompts). This is CCA architectural principle #2: programmatic hooks are the only reliable enforcement mechanism.